# Model 4: License Plate Detection + OCR
**YOLOv8 training on Kaggle GPU**

Detects and crops license plate regions.
EasyOCR then reads the plate text.
Class: `license_plate`

In [ ]:
!pip install ultralytics easyocr -q

In [ ]:
import os, yaml, shutil, cv2
import numpy as np
from ultralytics import YOLO
import easyocr

# ── Configuration ──────────────────────────────────────────────
DATASET_PATH = '/kaggle/input/your-plate-dataset'   # <-- CHANGE THIS
PROJECT_NAME = 'license_plate_detection'
MODEL_BASE   = 'yolov8n.pt'
EPOCHS       = 80
IMG_SIZE     = 640
BATCH_SIZE   = 16
# ───────────────────────────────────────────────────────────────

In [ ]:
data_yaml = os.path.join(DATASET_PATH, 'data.yaml')

if not os.path.exists(data_yaml):
    config = {
        'path': DATASET_PATH,
        'train': 'train/images',
        'val':   'val/images',
        'nc': 1,
        'names': ['license_plate']
    }
    data_yaml = '/kaggle/working/plate_data.yaml'
    with open(data_yaml, 'w') as f:
        yaml.dump(config, f)
    print('Created data.yaml')

with open(data_yaml) as f:
    print(f.read())

In [ ]:
model = YOLO(MODEL_BASE)

results = model.train(
    data    = data_yaml,
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    project = '/kaggle/working/runs',
    name    = PROJECT_NAME,
    device  = 0,
    patience= 25,
    save    = True,
    plots   = True,
    # License plate: rectangular, needs aspect ratio awareness
    scale   = 0.3,
    translate = 0.05,
    degrees = 3.0,        # plates rarely tilted much
    fliplr  = 0.5,
    mosaic  = 0.3,
    erasing = 0.2,        # simulate partial occlusion
    close_mosaic = 15,
)

In [ ]:
best_weights = f'/kaggle/working/runs/{PROJECT_NAME}/weights/best.pt'
model_best = YOLO(best_weights)
metrics = model_best.val(data=data_yaml, imgsz=IMG_SIZE)
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)

In [ ]:
# ── OCR pipeline test ───────────────────────────────────────────
def preprocess_plate(crop):
    """Enhance plate crop for better OCR accuracy."""
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    # Upscale for OCR
    h, w = gray.shape
    gray = cv2.resize(gray, (w*2, h*2), interpolation=cv2.INTER_CUBIC)
    # Denoise + sharpen
    gray = cv2.fastNlMeansDenoising(gray, h=10)
    kernel = np.array([[-1,-1,-1],[-1,9,-1],[-1,-1,-1]])
    gray = cv2.filter2D(gray, -1, kernel)
    # Adaptive threshold for variable lighting
    binary = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 11, 2
    )
    return binary

reader = easyocr.Reader(['en'], gpu=True)
print('EasyOCR reader initialised')

def read_plate(image_path, plate_model, ocr_reader):
    """Full pipeline: detect plate → crop → OCR."""
    img = cv2.imread(image_path)
    results = plate_model.predict(img, conf=0.4, verbose=False)
    plates = []
    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            crop = img[y1:y2, x1:x2]
            proc = preprocess_plate(crop)
            text_results = ocr_reader.readtext(
                proc,
                allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789',
                detail=1
            )
            plate_text = ''.join([
                t[1] for t in text_results if t[2] > 0.4
            ]).strip()
            plates.append({
                'bbox': (x1, y1, x2, y2),
                'confidence': float(box.conf),
                'plate_number': plate_text
            })
    return plates

# Test on a sample
import glob, random
samples = glob.glob(os.path.join(DATASET_PATH, 'val/images/*.jpg'))[:5]
if samples:
    test_img = random.choice(samples)
    detected = read_plate(test_img, model_best, reader)
    print('Test image:', test_img)
    for d in detected:
        print(f'  Plate: {d["plate_number"]}  conf={d["confidence"]:.2f}')
else:
    print('No val images found for test')

In [ ]:
output_path = '/kaggle/working/model4_license_plate.pt'
shutil.copy(best_weights, output_path)
print(f'Model saved: {output_path}')